# 02 — Optimization Foundations

## Objective
Derive common post-training objectives and verify small numeric examples with automatic differentiation.

## Background
Autoregressive models predict the next token from prior context. A training objective maps the difference between model behavior and desired behavior into a differentiable loss. SFT (Supervised Fine-Tuning), DPO (Direct Preference Optimization), and GRPO (Group Relative Policy Optimization) optimize different signals.

### SFT (Supervised Fine-Tuning)

$$\mathcal{L}_{\mathrm{SFT}}(\theta)=-\sum_{t=1}^{T}\log\pi_{\theta}(y_t\mid x,y_{<t})$$

The model parameters are $\theta$; $x$ is the prompt; $y_t$ is the target token at position $t$; and $T$ is the response length.

### LoRA (Low-Rank Adaptation)

$$W' = W + \frac{\alpha}{r}BA$$

$W$ is a frozen matrix; $A$ and $B$ are learned low-rank factors; $r$ is their rank; and $\alpha$ scales the update. QLoRA (Quantized Low-Rank Adaptation) additionally stores frozen weights in low precision.

### DPO (Direct Preference Optimization)

$$\mathcal L_{\mathrm{DPO}}=-\mathbb E_{(x,y_w,y_l)}\log\sigma\!\left[\beta\left(\log\frac{\pi_{\theta}(y_w\mid x)}{\pi_{\mathrm{ref}}(y_w\mid x)}-\log\frac{\pi_{\theta}(y_l\mid x)}{\pi_{\mathrm{ref}}(y_l\mid x)}\right)\right]$$

$y_w$ is the preferred response, $y_l$ the rejected response, $\pi_{\mathrm{ref}}$ the fixed reference model, $\beta$ a positive scale, and $\sigma$ the sigmoid function.

### GRPO (Group Relative Policy Optimization)

$$A_i=\frac{r_i-\overline r}{s_r+\varepsilon}$$

Each completion gets reward $r_i$; $\overline r$ and $s_r$ are the mean and standard deviation within its group. Advantage $A_i$ informs a policy-gradient update. This is the group-advantage term, not the complete GRPO objective.

### Check the Direct Preference Optimization (DPO) gradient

**Procedure.** Calculate a toy preference loss with PyTorch automatic differentiation and check the gradient signs.

**Reading the result.** A negative gradient for the chosen log-probability and positive gradient for the rejected one indicate the expected update direction; these numbers are illustrative. All example log-probabilities are nonpositive, as required for discrete response probabilities.

In [1]:
import torch
chosen = torch.tensor(-1.2, requires_grad=True)  # policy log-probability
chosen

tensor(-1.2000, requires_grad=True)

In [1]:
# Purpose: Calculate a toy preference loss with PyTorch automatic differentiation and check
#   the gradient signs.
# Result: A negative gradient for the chosen log-probability and positive gradient for the
#   rejected one indicate the expected update direction; these numbers are illustrative.
import torch
chosen = torch.tensor(-1.2, requires_grad=True)  # policy log-probability
rejected = torch.tensor(-1.5, requires_grad=True)
ref_chosen, ref_rejected, beta = -1.1, -1.2, 0.1
margin = beta*((chosen-ref_chosen)-(rejected-ref_rejected))
loss = -torch.nn.functional.logsigmoid(margin)
loss.backward()
print('DPO loss:',float(loss.detach()))
print('Gradient (chosen/rejected):',chosen.grad.item(),rejected.grad.item())
assert chosen.grad.item() < 0 and rejected.grad.item() > 0

DPO loss: 0.6831972002983093
Gradient (chosen/rejected): -0.04950001835823059 0.04950001835823059


### Normalize candidate rewards

**Procedure.** Center and scale four illustrative reward values to construct group-relative advantages.

**Reading the result.** The advantages have approximately zero mean. A zero-variance reward group contains no within-group ranking signal.

In [2]:
# Purpose: Center and scale four illustrative reward values to construct group-relative
#   advantages.
# Result: The advantages have approximately zero mean. A zero-variance reward group contains
#   no within-group ranking signal.
import numpy as np
rewards = np.array([1.,0.,1.,0.])
advantages = (rewards-rewards.mean())/(rewards.std()+1e-8)
print('GRPO-like group advantages:', advantages)
assert abs(advantages.mean())<1e-7

GRPO-like group advantages: [ 0.99999998 -0.99999998  0.99999998 -0.99999998]


## Q&A
**Question:** Why can GRPO obtain zero relative advantage despite high raw reward?

**Answer:** If every sample in a prompt group earns the same score, the standardized group advantage is zero.

### Reproducibility note
Each printed example belongs to one of three categories: an illustrative unit check, a source-backed inspection, or an opt-in model experiment. Only real executed experiments with source/model identifiers and saved outputs can support research findings.